## Loading packages <a class="anchor" id="Loading-packages"></a>

In [ ]:
from pathlib import Path
import os

# Locate the data by walking up from the notebook's working directory. Two layouts are supported:
#   1. <root>/data/spatial                          (self-contained cell2location folder; results go to <root>/results)
#   2. <root>/data/cell2Location/spatial            (MinorGland_paper repository; results go to <root>/results/cell2location)
def find_dirs(start=Path.cwd()):
    for p in [start, *start.parents]:
        if (p / "data" / "spatial").is_dir():
            return p / "data", p / "results"
        if (p / "data" / "cell2Location" / "spatial").is_dir():
            return p / "data" / "cell2Location", p / "results" / "cell2location"
    raise FileNotFoundError("Could not find data/spatial or data/cell2Location/spatial; set DATA_DIR and OUT_DIR manually.")

DATA_DIR, OUT_DIR = find_dirs()                  # DATA_DIR: inputs (read-only)
RESULTS_DIR = OUT_DIR / "models"                 # trained models and h5ad outputs
PROCESSED_DIR = OUT_DIR / "spatial_processed"    # merged / subset Visium objects written by this notebook
FIG_DIR = OUT_DIR / "figures"                    # relative savefig() calls land here
for d in (RESULTS_DIR, PROCESSED_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)
os.chdir(FIG_DIR)
print("DATA_DIR:", DATA_DIR)
print("OUT_DIR: ", OUT_DIR)

In [ ]:
# cell2location is not bundled with this repo. Install once in a fresh environment, e.g.:
#   pip install "scvi-tools" git+https://github.com/BayraktarLab/cell2location.git
# (a GPU is strongly recommended for the training cells)


In [ ]:
import scanpy as sc
import anndata
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

import cell2location
import scvi

import matplotlib as mpl
from matplotlib import rcParams
import matplotlib.pyplot as plt
import seaborn as sns

from matplotlib import rcParams
rcParams['pdf.fonttype'] = 42
import rich.pretty


Define where we save the results of our Slide1 analysis:

In [ ]:
from os import makedirs
from os.path import exists

results_folder = str(RESULTS_DIR / 'Sub_Focus')

# create paths and names to results folders for reference regression and cell2location models
slide4_ref_run_name = f'{results_folder}/all_reference_signatures'
slide4_run_name = f'{results_folder}/MinorInMajor_slide4_cell2location_map'

if not exists(slide4_ref_run_name):
  makedirs(slide4_ref_run_name)
if not exists(slide4_run_name):
  makedirs(slide4_run_name)




## Loading scRNA-seq reference data <a class="anchor" id="Loading-Visium-and-scRNA-seq-reference-data"></a>

load single cell data

In [ ]:
# Read data
adata_ref = sc.read(f'{DATA_DIR}/Ampullate_sc/Keep400_Major_Minor_new.h5ad')

In [ ]:
adata_ref.var['SYMBOL'] = adata_ref.var.index

In [ ]:
adata_ref.var

In [ ]:
adata_ref.obs

In [ ]:
CellTypes = ['Minor_ZoneA_MiSpabd', 'Minor_ZoneB_MiSpc','Minor_ZoneC']
adata_ref = adata_ref[adata_ref.obs["CellTypes"].isin (CellTypes)].copy()

In [ ]:
np.unique(adata_ref.obs.CellTypes)

In [ ]:
adata_raw = adata_ref.raw
adata_ref.X = adata_raw.X

In [ ]:
adata_ref.X.todense()

In [ ]:
del adata_ref.raw

#### Estimation of reference cell type signatures (NB regression) <a class="anchor" id="Estimation-of-reference-cell-type-signatures-(NB-regression)"></a>

The signatures are estimated from scRNA-seq data, accounting for batch effect, using a Negative binomial regression model.

<div class="alert alert-block alert-message">
<b>Preparing anndata.</b>
    
First, prepare anndata object for the regression model:
</div>

In [ ]:
# prepare anndata for the regression model
cell2location.models.RegressionModel.setup_anndata(adata=adata_ref,
                        # 10X reaction / sample / batch
                        batch_key='orig.ident',
                        # cell type, covariate used for constructing signatures
                        labels_key='CellTypes',
                        # multiplicative technical effects (platform)
                        categorical_covariate_keys=['Batch']
                       )

In [ ]:
adata_ref.obs[['_indices', '_scvi_batch', '_scvi_labels']]

In [ ]:
# create the regression model
from cell2location.models import RegressionModel
mod = RegressionModel(adata_ref)

# view anndata_setup as a sanity check
mod.view_anndata_setup()

<div class="alert alert-block alert-message">
<b>Training model.</b>
    
Now we train the model to estimate the reference cell type signatures.
    
Note that to achieve convergence on your data (=to get stabilization of the loss) you may need to increase `max_epochs=250` (See below).
    
Also note that here we are using `batch_size=2500` which is much larger than scvi-tools default and perform training on all cells in the data (`train_size=1`) - both parameters are defaults.
</div>

In [ ]:
# Reuse the saved regression model if it already exists (delete the folder to retrain)
if (Path(slide4_ref_run_name) / "model.pt").exists():
    mod = RegressionModel.load(slide4_ref_run_name, adata_ref)
    print("Loaded cached regression model from", slide4_ref_run_name)
    TRAINED_REF = False
else:
    mod.train(max_epochs=1800)
    TRAINED_REF = True


<div class="alert alert-block alert-message">
<b>Determine if the model needs more training.</b>
</div>

Here, we plot ELBO loss history during training, removing first 20 epochs from the plot.
This plot should have a decreasing trend and level off by the end of training. If it is still decreasing, increase `max_epochs`.

In [ ]:
if TRAINED_REF:
    mod.plot_history(20)


In [ ]:
# In this section, we export the estimated cell abundance (summary of the posterior distribution).
adata_ref = mod.export_posterior(
    adata_ref, sample_kwargs={'num_samples': 1000, 'batch_size': 2500},
    export_slot = 'mod',
    add_to_varm = ['means', 'stds', 'q05', 'q95']
)

In [ ]:

# Save model
mod.save(f"{slide4_ref_run_name}", overwrite=True)

# Save anndata object with results
adata_file = f"{slide4_ref_run_name}/sc_Minor.h5ad"
adata_ref.write(adata_file)
adata_file

<div class="alert alert-block alert-message">
<b>Examine QC plots.</b>
</div>

1. Reconstruction accuracy to assess if there are any issues with inference. This 2D histogram plot should have most observations along a noisy diagonal.

2. The estimated expression signatures are distinct from mean expression in each cluster because of batch effects. For scRNA-seq datasets which do not suffer from batch effect (this dataset does), cluster average expression can be used instead of estimating signatures with a model. When this plot is very different from a diagonal plot (e.g. very low values on Y-axis, density everywhere) it indicates problems with signature estimation.

In [ ]:
mod.plot_QC()

<div class="alert alert-block alert-message">
<b>Extracting reference cell types signatures as a pd.DataFrame.</b>
    
All parameters of the a Negative Binomial regression model are exported into reference anndata object, however for spatial mapping we just need the estimated expression of every gene in every cell type. Here we extract that from standard output:
</div>

In [ ]:
# export estimated expression in each cluster
if 'means_per_cluster_mu_fg' in adata_ref.varm.keys():
    inf_aver = adata_ref.varm['means_per_cluster_mu_fg'][[f'means_per_cluster_mu_fg_{i}'
                                    for i in adata_ref.uns['mod']['factor_names']]].copy()
else:
    inf_aver = adata_ref.var[[f'means_per_cluster_mu_fg_{i}'
                                    for i in adata_ref.uns['mod']['factor_names']]].copy()
inf_aver.columns = adata_ref.uns['mod']['factor_names']
inf_aver.iloc[0:10, 0:15]

## Spatial data analysis -- slide_4<a class="anchor" id="Loading-Visium-and-scRNA-seq-reference-data"></a>

In [ ]:
from os import makedirs
from os.path import exists

results_folder = str(RESULTS_DIR / 'Sub_Focus')

# create paths and names to results folders for reference regression and cell2location models
slide4_ref_run_name = f'{results_folder}/all_reference_signatures'
slide4_run_name = f'{results_folder}/MinorInMajor_slide4_cell2location_map'

if not exists(slide4_ref_run_name):
  makedirs(slide4_ref_run_name)
if not exists(slide4_run_name):
  makedirs(slide4_run_name)

**load ST data**

In [ ]:
adata_vis = sc.read_visium(f'{DATA_DIR}/spatial/P18163_1004', count_file='filtered_feature_bc_matrix.h5', load_images=True)


In [ ]:
glands_barcodes = pd.read_csv(f'{DATA_DIR}/spatial/P18163_1004_Silkglands.csv')
glands_barcodes

In [ ]:
adata_vis.var_names_make_unique()
adata_vis

In [ ]:
adata_vis.obs

In [ ]:
adata_vis.obs["cell_ids"] = adata_vis.obs.index
adata_vis_2 = pd.merge(adata_vis.obs, glands_barcodes, left_on = "cell_ids", right_on = "Barcode", how = "left")
adata_vis.obs = adata_vis_2
adata_vis.obs.set_index('cell_ids', drop=True, inplace=True)

In [ ]:

Glands = ['Minor']
adata_spatial = adata_vis[adata_vis.obs["Silk glands"].isin(Glands)]

fig, ax = plt.subplots()
sc.pl.spatial(adata_spatial, color='Silk glands', ax=ax, show=False,palette={'Minor': '#FFB14EFF'})

fig.savefig("Minor_spatial_plot_1.pdf")

plt.show()


In [ ]:
sc.pp.neighbors(adata_vis)

palette = {
    'Major': "#CCEBC5FF",
    'Minor': "#FD9AECFF",
    'Aggregate': '#FF91C4FF',
    'Tubuliform': '#C935B5FF',
    'Flagelliform': '#FA8775FF'
}

fig, ax = plt.subplots()
sc.pl.spatial(adata_vis, color='Silk glands', groups=['Minor','Major'],ax=ax, show=False,palette=palette)

plt.legend()

fig.savefig("Minor_spatial_plot_2.pdf")

plt.show()


In [ ]:
sc.pp.neighbors(adata_vis)

palette = {
    'Major': '#FFD700FF',
    'Minor': '#FFB14EFF',
    'Aggregate': '#FF91C4FF',
    'Tubuliform': '#C935B5FF',
    'Flagelliform': '#FA8775FF'
}

fig, ax = plt.subplots()
sc.pl.spatial(adata_vis, color='Silk glands', groups=['Minor'],ax=ax, show=False, palette=palette)

plt.legend()

fig.savefig("Minor_spatial_plot_3.pdf")

plt.show()

In [ ]:
Glands = ['Major']
adata_spatial = adata_vis[adata_vis.obs["Silk glands"].isin (Glands)]

In [ ]:
adata_spatial.obs

In [ ]:
adata_spatial.obs["Silk glands"].unique()
adata_spatial

In [ ]:
adata_spatial.var.set_index('gene_ids', drop=True, inplace=True)

In [ ]:
adata_spatial.var['SYMBOL'] = adata_spatial.var.index

In [ ]:
adata_spatial.var

In [ ]:
adata_spatial.obs['sample'] = list(adata_spatial.uns['spatial'].keys())[0]

In [ ]:
# find mitochondria-encoded (MT) genes
adata_spatial.var['MT_gene'] = [gene.startswith('MT-') for gene in adata_spatial.var['SYMBOL']]

# remove MT genes for spatial mapping (keeping their counts in the object)
adata_spatial.obsm['MT'] = adata_spatial[:, adata_spatial.var['MT_gene'].values].X.toarray()
adata_spatial = adata_spatial[:, ~adata_spatial.var['MT_gene'].values]

#### Cell2location: spatial mapping -- slide_4 <a class="anchor" id="Cell2location:-spatial-mapping"></a>

<div class="alert alert-block alert-message">
<b>Find shared genes and prepare anndata.</b>
Subset both anndata and reference signatures:
</div>

In [ ]:
# load reference single cell signature
adata_file = f"{slide4_ref_run_name}/sc_Minor.h5ad"
adata_ref = sc.read_h5ad(adata_file)
mod = cell2location.models.RegressionModel.load(f"{slide4_ref_run_name}", adata_ref)

In [ ]:
# export estimated expression in each cluster
if 'means_per_cluster_mu_fg' in adata_ref.varm.keys():
    inf_aver = adata_ref.varm['means_per_cluster_mu_fg'][[f'means_per_cluster_mu_fg_{i}'
                                    for i in adata_ref.uns['mod']['factor_names']]].copy()
else:
    inf_aver = adata_ref.var[[f'means_per_cluster_mu_fg_{i}'
                                    for i in adata_ref.uns['mod']['factor_names']]].copy()
inf_aver.columns = adata_ref.uns['mod']['factor_names']
inf_aver.iloc[0:5, 0:15]

In [ ]:
#silks = pd.read_csv(f'{DATA_DIR}/Major_sc/silk.present.csv')
#silks['geneName']

In [ ]:
Marker_Genes = pd.read_csv(f'{DATA_DIR}/minor/Sub_Recluster_allmarkers_join_geneinfo_0.15.tsv', sep = '\t')
Marker_Genes['gene']

In [ ]:
# find shared genes and subset both anndata and reference signatures
intersect = np.intersect1d(adata_spatial.var.index, inf_aver.index)
#intersect = np.intersect1d(intersect, silks['geneName'])
intersect = np.intersect1d(intersect, Marker_Genes['gene'])
adata_spatial = adata_spatial[:, intersect].copy()
inf_aver = inf_aver.loc[intersect, :].copy()

# prepare anndata for cell2location model
cell2location.models.Cell2location.setup_anndata(adata=adata_spatial, batch_key="sample")

In [ ]:
# create and train the model
mod = cell2location.models.Cell2location(
    adata_spatial, cell_state_df=inf_aver,
    # the expected average cell abundance: tissue-dependent hyper-prior which can be estimated from paired histology:
    N_cells_per_location=5,
    # hyperparameter controlling normalisation of within-experiment variation in RNA detection: high technical variability: 20; low technical variability: 200
    detection_alpha=20,
    detection_mean_per_sample = False
)
mod.view_anndata_setup()

<div class="alert alert-block alert-message">
<b>Training cell2location:</b>
</div>

In [ ]:
# Reuse the saved cell2location model if it already exists (delete the folder to retrain)
if (slide4_run_name and (Path(slide4_run_name) / "model.pt").exists()):
    mod = cell2location.models.Cell2location.load(slide4_run_name, adata_spatial)
    print("Loaded cached model from", slide4_run_name)
else:
    mod.train(max_epochs=5500,
              # train using full data (batch_size=None)
              batch_size=None,
              # use all data points in training because we need to estimate cell abundance at all locations
              train_size=1)

    # plot ELBO loss history during training, removing first 100 epochs from the plot
    mod.plot_history(1000)
    plt.legend(labels=['full data training']);


<div class="alert alert-block alert-message">
<b>Exporting estimated posterior distributions of cell abundance and saving results:</b>
</div>

In [ ]:
# In this section, we export the estimated cell abundance (summary of the posterior distribution).
adata_spatial = mod.export_posterior(
    adata_spatial, sample_kwargs={'num_samples': 1000, 'batch_size': mod.adata.n_obs}
)


In [ ]:
# Save model
mod.save(f"{slide4_run_name}", overwrite=True)

# mod = cell2location.models.Cell2location.load(f"{run_name}", adata_minor)

# Save anndata object with results
adata_file = f"{slide4_run_name}/sp_MinorInMajor.h5ad"
adata_spatial.write(adata_file)
adata_file

<div class="alert alert-block alert-message">
<b>Assessing mapping quality.</b>
Examine reconstruction accuracy to assess if there are any issues with mapping.
The plot should be roughly diagonal, strong deviations will signal problems that need to be investigated.
</div>

In [ ]:
mod.plot_QC()

In [ ]:
fig = mod.plot_spatial_QC_across_batches()

#### Visualising cell abundance in spatial coordinates -- slide_4 <a class="anchor" id="Visualising-cell-abundance-in-spatial-coordinates"></a>

<div class="alert alert-info">
Note
    
We use 5% quantile of the posterior distribution, representing the value of cell abundance that the model has high confidence in (aka 'at least this amount is present').
</div>

In [ ]:
# load reference single cell signature
adata_file = f"{slide4_ref_run_name}/sc_Minor.h5ad"
adata_ref = sc.read_h5ad(adata_file)
mod = cell2location.models.RegressionModel.load(f"{slide4_ref_run_name}", adata_ref)


In [ ]:
# export estimated expression in each cluster
if 'means_per_cluster_mu_fg' in adata_ref.varm.keys():
    inf_aver = adata_ref.varm['means_per_cluster_mu_fg'][[f'means_per_cluster_mu_fg_{i}'
                                    for i in adata_ref.uns['mod']['factor_names']]].copy()
else:
    inf_aver = adata_ref.var[[f'means_per_cluster_mu_fg_{i}'
                                    for i in adata_ref.uns['mod']['factor_names']]].copy()
inf_aver.columns = adata_ref.uns['mod']['factor_names']
inf_aver.iloc[0:5, 0:15]

In [ ]:
adata_file = f"{slide4_run_name}/sp_MinorInMajor.h5ad"
adata_spatial = sc.read_h5ad(adata_file)
mod = cell2location.models.Cell2location.load(f"{slide4_run_name}", adata_spatial)

In [ ]:
adata_spatial

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import scanpy as sc
import matplotlib as mpl
from PIL import Image
import os
import shutil
import math

Minor_Slide4_celltypes

In [ ]:
# add 5% quantile, representing confident cell abundance, 'at least this amount is present',
# to adata.obs with nice names for plotting
adata_spatial.obs[adata_spatial.uns['mod']['factor_names']] = adata_spatial.obsm['q05_cell_abundance_w_sf']

In [ ]:
# plot in spatial coordinates
plt.figure(dpi = 300)
with mpl.rc_context({'axes.facecolor':  'black',
                     'figure.figsize': [4.5, 5]}):

    sc.pl.spatial(adata_spatial, cmap='magma',#magma viridis
                  # show all Minor cell types
                  color=['Minor_ZoneA_MiSpabd',
                         'Minor_ZoneB_MiSpc', 'Minor_ZoneC'
                         ],
                  ncols=4, size=1.3,
                  img_key='hires',
                  # limit color scale at 99.2% quantile of cell abundance
                  #vmin=0, vmax=6,
                  save = 'MinorInMajor_Slide4_celltypes_2.pdf'
                 )

Minor_Slide4_Genes_1

In [ ]:
# plot in spatial coordinates
with mpl.rc_context({'axes.facecolor':  'black',
                     'figure.figsize': [4.5, 5]}):

    sc.pl.spatial(adata_spatial, cmap='magma',
                  # show genes
                  color=['MiSpa','MiSpb','MiSpd','MiSpc',
                  'Spidroin3', 'LASC013229', 'xync', 'pnliprp1'],
                  ncols=4, size=1.3,
                  img_key='hires',
                  vmin=0, vmax=50,
                  save = 'Minor_Slide4_Genes_1.pdf'
                 )

Minor_Slide4_Genes_2

In [ ]:
# plot in spatial coordinates
with mpl.rc_context({'axes.facecolor':  'black',
                     'figure.figsize': [4.5, 5]}):

    sc.pl.spatial(adata_spatial, cmap='magma',
                  # show Misp genes
                  color=['MiSpa','MiSpb','MiSpd','MiSpc',
                  'Spidroin3', 'LASC013229', 'xync', 'pnliprp1'
                         ],
                  ncols=4, size=1.3,
                  img_key='hires',
                  vmin=0, vmax=5,
                  save = 'Minor_Slide4_Genes_2.pdf'
                 )

with scale bar

In [ ]:
temp_dir = 'temp_images'
os.makedirs(temp_dir, exist_ok=True)

original_cwd = os.getcwd()

colors = ['MiSpa','MiSpb','MiSpd','MiSpc',
          'Spidroin3', 'LASC013229', 'xync', 'pnliprp1'
          ]

os.chdir(temp_dir)
for i, color in enumerate(colors):
    sc.pl.spatial(adata_spatial, cmap='magma',
                  color=color,
                  size=1.3,
                  img_key='hires',
                  vmin=0, vmax=5,
                  save=f'spatial_{i}.png')

os.chdir(original_cwd)

In [ ]:
num_colors = len(colors)
ncols = 4
nrows = math.ceil(num_colors / ncols)

scale_bar_length = 100
scale_bar_text = '100 μm'


fig, axes = plt.subplots(nrows, ncols, figsize=(18, 5 * nrows), dpi=300, constrained_layout=True)
axes = axes.flatten()
for i, ax in enumerate(axes):
    if i < num_colors:
        img_path = os.path.join(temp_dir, f'figures/showspatial_{i}.png')
        img = Image.open(img_path)
        ax.imshow(img)

        scale_bar = patches.Rectangle((50, 50), scale_bar_length, 10, linewidth=1, edgecolor='black', facecolor='black')
        ax.add_patch(scale_bar)

        ax.text(50 + scale_bar_length / 2, 70, scale_bar_text, color='black', ha='center', va='top')
    else:

        ax.axis('of')

plt.show()


fig.savefig('Minor_Slide4_Genes_2_with_scalebar.pdf', dpi=300, bbox_inches='tight')

import shutil
shutil.rmtree(temp_dir)

In [ ]:
# Now we use cell2location plotter that allows showing multiple cell types in one panel
from cell2location.plt import plot_spatial

# select clusters
clust_labels = ['Minor_ZoneA_MiSpabd','Minor_ZoneB_MiSpc', 'Minor_ZoneC','Minor_Duct_1', 'Minor_Duct_2']
clust_col = ['' + str(i) for i in clust_labels]

with mpl.rc_context({'figure.figsize': (15, 15)}):
    fig = plot_spatial(
        adata=adata_spatial,
        # labels to show on a plot
        color=clust_col, labels=clust_labels,
        show_img=False,
        # 'fast' (white background) or 'dark_background'
        style='fast',
        # limit color scale at 99.2% quantile of cell abundance
        max_color_quantile=0.992,
        # size of locations (adjust depending on figure size)
        circle_diameter=6,
        colorbar_position='right'
    )
    #fig.savefig('Major_Slide4_celltypes_2.pdf', dpi=300)

In [ ]:
with mpl.rc_context({'figure.figsize': (15, 15)}):
    fig = plot_spatial(
        adata=adata_spatial,
        # labels to show on a plot
        color=clust_col, labels=clust_labels,
        show_img=True,
        # 'fast' (white background) or 'dark_background'
        style='fast',
        # limit color scale at 99.2% quantile of cell abundance
        max_color_quantile=0.992,
        # size of locations (adjust depending on figure size)
        circle_diameter=6,
        colorbar_position='right'
    )
    #fig.savefig('Major_Slide4_celltypes_3.pdf', dpi=300)

# Merge 8 slides

In [ ]:
# Set paths to data and results used through the document:
sp_data_folder = f'{DATA_DIR}/spatial/'
results_folder = str(RESULTS_DIR / 'All_Slides_Minor_2')

regression_model_output = 'RegressionGeneBackgroundCoverageTorch'
reg_path = f'{results_folder}/regression_model/{regression_model_output}/'

In [ ]:
def read_and_qc(sample_name, path=sp_data_folder):
    r""" This function reads the data for one 10X spatial experiment into the anndata object.
    It also calculates QC metrics. Modify this function if required by your workflow.

    :param sample_name: Name of the sample
    :param path: path to data
    """

    adata = sc.read_visium(path + str(sample_name),
                           count_file='filtered_feature_bc_matrix.h5', load_images=True)
    adata.obs['sample'] = sample_name
    adata.var['SYMBOL'] = adata.var_names
    adata.var.rename(columns={'gene_ids': 'ENSEMBL'}, inplace=True)
    adata.var_names = adata.var['ENSEMBL']
    adata.var.drop(columns='ENSEMBL', inplace=True)

    # Calculate QC metrics
    from scipy.sparse import csr_matrix
    adata.X = adata.X.toarray()
    sc.pp.calculate_qc_metrics(adata, inplace=True)
    adata.X = csr_matrix(adata.X)
    adata.var['mt'] = [gene.startswith('mt-') for gene in adata.var['SYMBOL']]
    adata.obs['mt_frac'] = adata[:, adata.var['mt'].tolist()].X.sum(1).A.squeeze()/adata.obs['total_counts']

    # add sample name to obs names
    adata.obs["sample"] = [str(i) for i in adata.obs['sample']]
    adata.obs_names = adata.obs["sample"] \
                          + '_' + adata.obs_names
    adata.obs.index.name = 'spot_id'

    return adata

def select_slide(adata, s, s_col='sample'):
    r""" This function selects the data for one slide from the spatial anndata object.

    :param adata: Anndata object with multiple spatial experiments
    :param s: name of selected experiment
    :param s_col: column in adata.obs listing experiment name for each location
    """

    slide = adata[adata.obs[s_col].isin([s]), :]
    s_keys = list(slide.uns['spatial'].keys())
    s_spatial = np.array(s_keys)[[s in k for k in s_keys]][0]

    slide.uns['spatial'] = {s_spatial: slide.uns['spatial'][s_spatial]}

    return slide



In [ ]:
#######################
# Read the list of spatial experiments
#sample_data = pd.read_csv(sp_data_folder + 'Visium_mouse.csv')
sample_data = ['P18163_1001', 'P18163_1002', 'P18163_1003', 'P18163_1004', 'P18163_1005', 'P22555_1001', 'P22555_1002', 'P22555_1003']

# Read the data into anndata objects
slides = []
for i in sample_data:
    slides.append(read_and_qc(i, path=sp_data_folder))


In [ ]:
# Combine anndata objects together
adata = slides[0].concatenate(
    slides[1:],
    batch_key="sample",
    uns_merge="unique",
    batch_categories=sample_data,
    index_unique=None
)

In [ ]:
adata.obs

In [ ]:
import matplotlib as mpl
from matplotlib import rcParams
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# PLOT QC FOR EACH SAMPLE
fig, axs = plt.subplots(len(slides), 4, figsize=(15, 4*len(slides)-4))
for i, s in enumerate(adata.obs['sample'].unique()):
    #fig.suptitle('Covariates for filtering')

    slide = select_slide(adata, s)
    sns.distplot(slide.obs['total_counts'],
                 kde=False, ax = axs[i, 0])
    axs[i, 0].set_xlim(0, adata.obs['total_counts'].max())
    axs[i, 0].set_xlabel(f'total_counts | {s}')

    sns.distplot(slide.obs['total_counts']\
                 [slide.obs['total_counts']<20000],
                 kde=False, bins=40, ax = axs[i, 1])
    axs[i, 1].set_xlim(0, 20000)
    axs[i, 1].set_xlabel(f'total_counts | {s}')

    sns.distplot(slide.obs['n_genes_by_counts'],
                 kde=False, bins=60, ax = axs[i, 2])
    axs[i, 2].set_xlim(0, adata.obs['n_genes_by_counts'].max())
    axs[i, 2].set_xlabel(f'n_genes_by_counts | {s}')

    sns.distplot(slide.obs['n_genes_by_counts']\
                 [slide.obs['n_genes_by_counts']<6000],
                 kde=False, bins=60, ax = axs[i, 3])
    axs[i, 3].set_xlim(0, 6000)
    axs[i, 3].set_xlabel(f'n_genes_by_counts | {s}')

plt.tight_layout()

In [ ]:
glands_barcodes_1 = pd.read_csv(f'{DATA_DIR}/spatial/P18163_1001_Silkglands.csv')
glands_barcodes_1['Barcode'] = 'P18163_1001_' + glands_barcodes_1['Barcode'].astype(str)

In [ ]:
glands_barcodes_2 = pd.read_csv(f'{DATA_DIR}/spatial/P18163_1002_Silkglands.csv')
glands_barcodes_2['Barcode'] = 'P18163_1002_' + glands_barcodes_2['Barcode'].astype(str)

In [ ]:
glands_barcodes_3 = pd.read_csv(f'{DATA_DIR}/spatial/P18163_1003_Silkglands.csv')
glands_barcodes_3['Barcode'] = 'P18163_1003_' + glands_barcodes_3['Barcode'].astype(str)
glands_barcodes_3['Silk glands'] = glands_barcodes_3['Silk glands'].replace('ampullate', 'Major')
print(glands_barcodes_3['Silk glands'].unique())

In [ ]:
glands_barcodes_4 = pd.read_csv(f'{DATA_DIR}/spatial/P18163_1004_Silkglands.csv')
glands_barcodes_4['Barcode'] = 'P18163_1004_' + glands_barcodes_4['Barcode'].astype(str)

In [ ]:
glands_barcodes_5 = pd.read_csv(f'{DATA_DIR}/spatial/P18163_1005_Silkglands.csv')
glands_barcodes_5['Barcode'] = 'P18163_1005_' + glands_barcodes_5['Barcode'].astype(str)

In [ ]:
glands_barcodes_6 = pd.read_csv(f'{DATA_DIR}/spatial/P22555_1001_Silkglands.csv')
glands_barcodes_6['Barcode'] = 'P22555_1001_' + glands_barcodes_6['Barcode'].astype(str)

In [ ]:
glands_barcodes_7 = pd.read_csv(f'{DATA_DIR}/spatial/P22555_1002_Silkglands.csv')
glands_barcodes_7['Barcode'] = 'P22555_1002_' + glands_barcodes_7['Barcode'].astype(str)

In [ ]:
glands_barcodes_8 = pd.read_csv(f'{DATA_DIR}/spatial/P22555_1003_Silkglands.csv')
glands_barcodes_8['Barcode'] = 'P22555_1003_' + glands_barcodes_8['Barcode'].astype(str)

In [ ]:
glands_barcodes = [glands_barcodes_1, glands_barcodes_2, glands_barcodes_3, glands_barcodes_4, glands_barcodes_5, glands_barcodes_6, glands_barcodes_7, glands_barcodes_8]

In [ ]:
glands_barcodes = pd.concat(glands_barcodes, ignore_index=True)

In [ ]:
glands_barcodes

In [ ]:
adata_vis = adata

In [ ]:
adata_vis.obs

In [ ]:
adata_vis.write_h5ad(PROCESSED_DIR / "all_8_slides_data.h5ad")

In [ ]:
#adata_vis.obs = adata_vis.obs.drop("spot_id", axis=1)

In [ ]:
adata_vis_2 = pd.merge(adata_vis.obs, glands_barcodes, left_on = "spot_id", right_on = "Barcode", how = "left")

In [ ]:
adata_vis.obs = adata_vis_2
adata_vis.obs.set_index('Barcode', drop=True, inplace=True)

In [ ]:
adata_minor = adata_vis[adata_vis.obs["Silk glands"] == 'Minor']

In [ ]:
adata_minor.var['SYMBOL'] = adata_minor.var.index

In [ ]:
# find mitochondria-encoded (MT) genes
adata_minor.var['MT_gene'] = [gene.startswith('MT-') for gene in adata_minor.var['SYMBOL']]

# remove MT genes for spatial mapping (keeping their counts in the object)
adata_minor.obsm['MT'] = adata_minor[:, adata_minor.var['MT_gene'].values].X.toarray()
adata_minor = adata_minor[:, ~adata_minor.var['MT_gene'].values]

In [ ]:
adata_minor.write_h5ad(PROCESSED_DIR / "all_8_slides_minor_data.h5ad")

In [ ]:
adata_major = adata_vis[adata_vis.obs["Silk glands"] == 'Major']
adata_major.var['SYMBOL'] = adata_major.var.index
# find mitochondria-encoded (MT) genes
adata_major.var['MT_gene'] = [gene.startswith('MT-') for gene in adata_major.var['SYMBOL']]

# remove MT genes for spatial mapping (keeping their counts in the object)
adata_major.obsm['MT'] = adata_major[:, adata_major.var['MT_gene'].values].X.toarray()
adata_major = adata_major[:, ~adata_major.var['MT_gene'].values]
adata_major.write_h5ad(PROCESSED_DIR / "all_8_slides_major_data.h5ad")

In [ ]:
adata_major.obs

In [ ]:
adata_minor_major = adata_vis[adata_vis.obs["Silk glands"].isin(['Minor', 'Major'])]
adata_minor_major.var['SYMBOL'] = adata_minor_major.var.index
# find mitochondria-encoded (MT) genes
adata_minor_major.var['MT_gene'] = [gene.startswith('MT-') for gene in adata_minor_major.var['SYMBOL']]

# remove MT genes for spatial mapping (keeping their counts in the object)
adata_minor_major.obsm['MT'] = adata_minor_major[:, adata_minor_major.var['MT_gene'].values].X.toarray()
adata_minor_major = adata_minor_major[:, ~adata_minor_major.var['MT_gene'].values]
adata_minor_major.write_h5ad(PROCESSED_DIR / "all_8_slides_minor_major_data.h5ad")

In [ ]:
adata_minor_major.obs

### Minor and Major Spots <a class="anchor" id="Cell2location:-spatial-mapping"></a>

<div class="alert alert-block alert-message">
<b>Find shared genes and prepare anndata.</b>
Subset both anndata and reference signatures:
</div>

In [ ]:
#adata_allSlides = sc.read_h5ad(f"{DATA_DIR}/spatial/all_8_slides_data.h5ad")

pal_pastel2 = c(Minor = "#FD9AECFF",
                "Aggregate duct" =  "#D2D2D2FF",
                Major =  "#CCEBC5FF",
                Flagelliform =  "#FBB4AEFF",
                Tubuliform =  "#738DC3FF",
                Aciniform  =     "#FED9A6FF",
                Aggregate =          "#DECBE4FF",
                Pyriform = "#FFFF77FF",
                OtherTissue = "#D2D2D233")   

Slide1

In [ ]:
allSlides_slide_1 = sc.read_visium(f'{DATA_DIR}/spatial/P18163_1001', count_file='filtered_feature_bc_matrix.h5', load_images=True)
glands_barcodes_1 = pd.read_csv(f'{DATA_DIR}/spatial/P18163_1001_Silkglands.csv')

In [ ]:
allSlides_slide_1.var_names_make_unique()

allSlides_slide_1.obs["cell_ids"] = allSlides_slide_1.obs.index
allSlides_slide_1_2 = pd.merge(allSlides_slide_1.obs, glands_barcodes_1, left_on = "cell_ids", right_on = "Barcode", how = "left")
allSlides_slide_1.obs = allSlides_slide_1_2
allSlides_slide_1.obs.set_index('cell_ids', drop=True, inplace=True)

In [ ]:
allSlides_slide_1.obs

In [ ]:
allSlides_slide_1.obs["Silk glands"].unique()

In [ ]:
sc.pp.neighbors(allSlides_slide_1)

palette = {
    'Minor': "#FD9AECFF",
    'Major': "#CCEBC5FF",
    'Flagelliform': "#FBB4AEFF",
    'Tubuliform': "#738DC3FF",
    'Aggregate ducts': "#D2D2D2FF",
    'Pyriform': "#FFFF77FF",
    'Aggregate gland': "#D2D2D2FF",
    'Aciniform': "#FED9A6FF"
}

fig, ax = plt.subplots()
sc.pl.spatial(allSlides_slide_1, color='Silk glands', groups=['Minor','Major'],ax=ax, show=False, palette=palette)

plt.legend()

fig.savefig("allSlides_slide_1.pdf")

plt.show()

Slide2

In [ ]:
allSlides_slide_2 = sc.read_visium(f'{DATA_DIR}/spatial/P18163_1002', count_file='filtered_feature_bc_matrix.h5', load_images=True)
glands_barcodes_2 = pd.read_csv(f'{DATA_DIR}/spatial/P18163_1002_Silkglands.csv')

In [ ]:
allSlides_slide_2.var_names_make_unique()

allSlides_slide_2.obs["cell_ids"] = allSlides_slide_2.obs.index
allSlides_slide_2_2 = pd.merge(allSlides_slide_2.obs, glands_barcodes_2, left_on = "cell_ids", right_on = "Barcode", how = "left")
allSlides_slide_2.obs = allSlides_slide_2_2
allSlides_slide_2.obs.set_index('cell_ids', drop=True, inplace=True)

In [ ]:
allSlides_slide_2.obs["Silk glands"].unique()

In [ ]:
sc.pp.neighbors(allSlides_slide_2)

palette = {
    'Minor': "#FD9AECFF",
    'Major': "#CCEBC5FF",
    'Aggregate gland': "#D2D2D2FF",
    'Pyriform': "#FFFF77FF",
    'Aciniform': "#FED9A6FF",
    'Aggregate duct': "#D2D2D2FF",
    'Flagelliform': "#FBB4AEFF",
    'Tubuliform': "#738DC3FF",
    'unsure': "#D2D2D233"
}

fig, ax = plt.subplots()
sc.pl.spatial(allSlides_slide_2, color='Silk glands', groups=['Minor','Major'],ax=ax, show=False, palette=palette)

plt.legend()

fig.savefig("allSlides_slide_2.pdf")

plt.show()

Slide3

In [ ]:
allSlides_slide_3 = sc.read_visium(f'{DATA_DIR}/spatial/P18163_1003', count_file='filtered_feature_bc_matrix.h5', load_images=True)
glands_barcodes_3 = pd.read_csv(f'{DATA_DIR}/spatial/P18163_1003_Silkglands.csv')
glands_barcodes_3['Silk glands'] = glands_barcodes_3['Silk glands'].replace('ampullate', 'Major')

In [ ]:
allSlides_slide_3.var_names_make_unique()
allSlides_slide_3.var_names_make_unique()
allSlides_slide_3.obs["cell_ids"] = allSlides_slide_3.obs.index
allSlides_slide_3_2 = pd.merge(allSlides_slide_3.obs, glands_barcodes_3, left_on = "cell_ids", right_on = "Barcode", how = "left")
allSlides_slide_3.obs = allSlides_slide_3_2
allSlides_slide_3.obs.set_index('cell_ids', drop=True, inplace=True)

In [ ]:
allSlides_slide_3.obs["Silk glands"].unique()

In [ ]:
sc.pp.neighbors(allSlides_slide_3)

palette = {
    'Major': "#CCEBC5FF",
    'Pyriform': "#FFFF77FF",
    'Aggregate gland': "#D2D2D2FF",
    'Aciniform': "#FED9A6FF",
    'Flagelliform': "#FBB4AEFF",
    'Tubuliform': "#738DC3FF"
}

fig, ax = plt.subplots()
sc.pl.spatial(allSlides_slide_3, color='Silk glands', groups=['Minor','Major'],ax=ax, show=False, palette=palette)

plt.legend()

fig.savefig("allSlides_slide_3.pdf")

plt.show()

Slide4

In [ ]:
allSlides_slide_4 = sc.read_visium(f'{DATA_DIR}/spatial/P18163_1004', count_file='filtered_feature_bc_matrix.h5', load_images=True)
glands_barcodes_4 = pd.read_csv(f'{DATA_DIR}/spatial/P18163_1004_Silkglands.csv')

In [ ]:
allSlides_slide_4.var_names_make_unique()

allSlides_slide_4.obs["cell_ids"] = allSlides_slide_4.obs.index
allSlides_slide_4_2 = pd.merge(allSlides_slide_4.obs, glands_barcodes_4, left_on = "cell_ids", right_on = "Barcode", how = "left")
allSlides_slide_4.obs = allSlides_slide_4_2
allSlides_slide_4.obs.set_index('cell_ids', drop=True, inplace=True)

In [ ]:
allSlides_slide_4.obs["Silk glands"].unique()

In [ ]:
allSlides_slide_4.obs

In [ ]:
sc.pp.neighbors(allSlides_slide_4)

palette = {
    'Minor': "#FD9AECFF",
    'Major': "#CCEBC5FF",
    'Flagelliform': "#FBB4AEFF",
    'Tubuliform': "#738DC3FF",
    'Aggregate': "#DECBE4FF"
}

fig, ax = plt.subplots()
sc.pl.spatial(allSlides_slide_4, color='Silk glands', groups=['Minor','Major'],ax=ax, show=False, palette=palette)

plt.legend()

fig.savefig("allSlides_slide_4.pdf")

plt.show()

Slide5

In [ ]:
allSlides_slide_5 = sc.read_visium(f'{DATA_DIR}/spatial/P18163_1005', count_file='filtered_feature_bc_matrix.h5', load_images=True)
glands_barcodes_5 = pd.read_csv(f'{DATA_DIR}/spatial/P18163_1005_Silkglands.csv')

In [ ]:
allSlides_slide_5.var_names_make_unique()

allSlides_slide_5.obs["cell_ids"] = allSlides_slide_5.obs.index
allSlides_slide_5_2 = pd.merge(allSlides_slide_5.obs, glands_barcodes_5, left_on = "cell_ids", right_on = "Barcode", how = "left")
allSlides_slide_5.obs = allSlides_slide_5_2
allSlides_slide_5.obs.set_index('cell_ids', drop=True, inplace=True)

In [ ]:
allSlides_slide_5.obs["Silk glands"].unique()

In [ ]:
sc.pp.neighbors(allSlides_slide_5)

palette = {
    'Minor': "#FD9AECFF",
    'Major': "#CCEBC5FF",
    'Aggregate gland': "#D2D2D2FF",
    'Aciniform': "#FED9A6FF",
    'Aggregate duct': "#D2D2D2FF",
    'Flagelliform': "#FBB4AEFF",
    'Tubuliform': "#738DC3FF"
}

fig, ax = plt.subplots()
sc.pl.spatial(allSlides_slide_5, color='Silk glands', groups=['Minor','Major'],ax=ax, show=False, palette=palette)

plt.legend()

fig.savefig("allSlides_slide_5.pdf")

plt.show()

Slide6

In [ ]:
allSlides_slide_6 = sc.read_visium(f'{DATA_DIR}/spatial/P22555_1001', count_file='filtered_feature_bc_matrix.h5', load_images=True)
glands_barcodes_6 = pd.read_csv(f'{DATA_DIR}/spatial/P22555_1001_Silkglands.csv')

In [ ]:
allSlides_slide_6.var_names_make_unique()

allSlides_slide_6.obs["cell_ids"] = allSlides_slide_6.obs.index
allSlides_slide_6_2 = pd.merge(allSlides_slide_6.obs, glands_barcodes_6, left_on = "cell_ids", right_on = "Barcode", how = "left")
allSlides_slide_6.obs = allSlides_slide_6_2
allSlides_slide_6.obs.set_index('cell_ids', drop=True, inplace=True)

In [ ]:
allSlides_slide_6.obs["Silk glands"].unique()

In [ ]:
sc.pp.neighbors(allSlides_slide_6)

palette = {
    'Minor': "#FD9AECFF",
    'Major': "#CCEBC5FF",
    'Flagelliform': "#FBB4AEFF",
    'Tubuliform': "#738DC3FF",
    'Aggregate gland': "#D2D2D2FF",
    'Aciniform': "#FED9A6FF",
    'Aggregate duct': "#D2D2D2FF"
}

fig, ax = plt.subplots()
sc.pl.spatial(allSlides_slide_6, color='Silk glands', groups=['Minor','Major'],ax=ax, show=False, palette=palette)

plt.legend()

fig.savefig("allSlides_slide_6.pdf")

plt.show()

Slide7

In [ ]:
allSlides_slide_7 = sc.read_visium(f'{DATA_DIR}/spatial/P22555_1002', count_file='filtered_feature_bc_matrix.h5', load_images=True)
glands_barcodes_7 = pd.read_csv(f'{DATA_DIR}/spatial/P22555_1002_Silkglands.csv')

In [ ]:
allSlides_slide_7.var_names_make_unique()

allSlides_slide_7.obs["cell_ids"] = allSlides_slide_7.obs.index
allSlides_slide_7_2 = pd.merge(allSlides_slide_7.obs, glands_barcodes_7, left_on = "cell_ids", right_on = "Barcode", how = "left")
allSlides_slide_7.obs = allSlides_slide_7_2
allSlides_slide_7.obs.set_index('cell_ids', drop=True, inplace=True)

In [ ]:
allSlides_slide_7.obs["Silk glands"].unique()

In [ ]:
sc.pp.neighbors(allSlides_slide_7)

palette = {
    'Minor': "#FD9AECFF",
    'Major': "#CCEBC5FF",
    'Flagelliform': "#FBB4AEFF",
    'Tubuliform': "#738DC3FF",
    'Aggregate gland': "#D2D2D2FF",
    'Aciniform': "#FED9A6FF",
    'Aggregate duct': "#D2D2D2FF"
}

fig, ax = plt.subplots()
sc.pl.spatial(allSlides_slide_7, color='Silk glands', groups=['Minor','Major'],ax=ax, show=False, palette=palette)

plt.legend()

fig.savefig("allSlides_slide_7.pdf")

plt.show()

Slide8

In [ ]:
allSlides_slide_8 = sc.read_visium(f'{DATA_DIR}/spatial/P22555_1003', count_file='filtered_feature_bc_matrix.h5', load_images=True)
glands_barcodes_8 = pd.read_csv(f'{DATA_DIR}/spatial/P22555_1003_Silkglands.csv')

In [ ]:
allSlides_slide_8.var_names_make_unique()

allSlides_slide_8.obs["cell_ids"] = allSlides_slide_8.obs.index
allSlides_slide_8_2 = pd.merge(allSlides_slide_8.obs, glands_barcodes_8, left_on = "cell_ids", right_on = "Barcode", how = "left")
allSlides_slide_8.obs = allSlides_slide_8_2
allSlides_slide_8.obs.set_index('cell_ids', drop=True, inplace=True)

In [ ]:
allSlides_slide_8.obs["Silk glands"].unique()

In [ ]:
sc.pp.neighbors(allSlides_slide_8)

palette = {
    'Minor': "#FD9AECFF",
    'Major': "#CCEBC5FF",
    'Flagelliform': "#FBB4AEFF",
    'Tubuliform': "#738DC3FF",
    'Aggregate gland': "#D2D2D2FF"
}

fig, ax = plt.subplots()
sc.pl.spatial(allSlides_slide_8, color='Silk glands', groups=['Minor','Major'],ax=ax, show=False, palette=palette)

plt.legend()

fig.savefig("allSlides_slide_8.pdf")

plt.show()

### Cell2location: spatial mapping -- 8slides -- MinorInAmpullate <a class="anchor" id="Cell2location:-spatial-mapping"></a>

<div class="alert alert-block alert-message">
<b>Find shared genes and prepare anndata.</b>
Subset both anndata and reference signatures:
</div>

In [ ]:
from os import makedirs
from os.path import exists

results_folder = str(RESULTS_DIR / 'Sub_Focus')

# create paths and names to results folders for reference regression and cell2location models
slide4_ref_run_name = f'{results_folder}/all_reference_signatures'
MinorInAmpullate_All_slide_run_name = f'{results_folder}/MinorInAmpullate_AllSlide_cell2location_map'

if not exists(slide4_ref_run_name):
  makedirs(slide4_ref_run_name)
if not exists(MinorInAmpullate_All_slide_run_name):
  makedirs(MinorInAmpullate_All_slide_run_name)

In [ ]:
# load reference single cell signature
adata_file = f"{slide4_ref_run_name}/sc_Minor.h5ad"
adata_ref = sc.read_h5ad(adata_file)
mod = cell2location.models.RegressionModel.load(f"{slide4_ref_run_name}", adata_ref)


In [ ]:
# export estimated expression in each cluster
if 'means_per_cluster_mu_fg' in adata_ref.varm.keys():
    inf_aver = adata_ref.varm['means_per_cluster_mu_fg'][[f'means_per_cluster_mu_fg_{i}'
                                    for i in adata_ref.uns['mod']['factor_names']]].copy()
else:
    inf_aver = adata_ref.var[[f'means_per_cluster_mu_fg_{i}'
                                    for i in adata_ref.uns['mod']['factor_names']]].copy()
inf_aver.columns = adata_ref.uns['mod']['factor_names']
inf_aver.iloc[0:5, 0:15]

In [ ]:
Marker_Genes = pd.read_csv(f'{DATA_DIR}/minor/Sub_Recluster_allmarkers_join_geneinfo_0.15.tsv', sep = '\t')
Marker_Genes['gene']

In [ ]:
# find shared genes and subset both anndata and reference signatures
intersect = np.intersect1d(adata_minor_major.var.index, inf_aver.index)
#intersect = np.intersect1d(intersect, silks['geneName'])
intersect = np.intersect1d(intersect, Marker_Genes['gene'])
adata_minor_major = adata_minor_major[:, intersect].copy()
inf_aver = inf_aver.loc[intersect, :].copy()

# prepare anndata for cell2location model
cell2location.models.Cell2location.setup_anndata(adata=adata_minor_major, batch_key="sample")

In [ ]:
# create and train the model
mod = cell2location.models.Cell2location(
    adata_minor_major, cell_state_df=inf_aver,
    # the expected average cell abundance: tissue-dependent hyper-prior which can be estimated from paired histology:
    N_cells_per_location=5,
    # hyperparameter controlling normalisation of within-experiment variation in RNA detection: high technical variability: 20; low technical variability: 200
    detection_alpha=20,
    detection_mean_per_sample = False
)
mod.view_anndata_setup()

<div class="alert alert-block alert-message">
<b>Training cell2location:</b>
</div>

In [ ]:
# Reuse the saved cell2location model if it already exists (delete the folder to retrain)
if (MinorInAmpullate_All_slide_run_name and (Path(MinorInAmpullate_All_slide_run_name) / "model.pt").exists()):
    mod = cell2location.models.Cell2location.load(MinorInAmpullate_All_slide_run_name, adata_minor_major)
    print("Loaded cached model from", MinorInAmpullate_All_slide_run_name)
else:
    mod.train(max_epochs=5500,
              # train using full data (batch_size=None)
              batch_size=None,
              # use all data points in training because we need to estimate cell abundance at all locations
              train_size=1)

    # plot ELBO loss history during training, removing first 100 epochs from the plot
    mod.plot_history(1000)
    plt.legend(labels=['full data training']);


<div class="alert alert-block alert-message">
<b>Exporting estimated posterior distributions of cell abundance and saving results:</b>
</div>

In [ ]:
# In this section, we export the estimated cell abundance (summary of the posterior distribution).
adata_spatial = mod.export_posterior(
    adata_minor_major, sample_kwargs={'num_samples': 1000, 'batch_size': mod.adata.n_obs}
)

# Save model
mod.save(f"{MinorInAmpullate_All_slide_run_name}", overwrite=True)

# mod = cell2location.models.Cell2location.load(f"{run_name}", adata_minor_major)

# Save anndata object with results
adata_file = f"{MinorInAmpullate_All_slide_run_name}/sp.h5ad"
adata_minor_major.write(adata_file)
adata_file

<div class="alert alert-block alert-message">
<b>Assessing mapping quality.</b>
Examine reconstruction accuracy to assess if there are any issues with mapping.
The plot should be roughly diagonal, strong deviations will signal problems that need to be investigated.
</div>

In [ ]:
mod.plot_QC()

In [ ]:
fig = mod.plot_spatial_QC_across_batches()
fig.savefig("Minor_Major_spatial_QC_across_batches.pdf")

#### Visualising cell abundance in spatial coordinates <a class="anchor" id="Visualising-cell-abundance-in-spatial-coordinates"></a>

<div class="alert alert-info">
Note
    
We use 5% quantile of the posterior distribution, representing the value of cell abundance that the model has high confidence in (aka 'at least this amount is present').
</div>

In [ ]:
# load reference single cell signature
adata_file = f"{slide4_ref_run_name}/sc_Minor.h5ad"
adata_ref = sc.read_h5ad(adata_file)
mod = cell2location.models.RegressionModel.load(f"{slide4_ref_run_name}", adata_ref)


In [ ]:
# export estimated expression in each cluster
if 'means_per_cluster_mu_fg' in adata_ref.varm.keys():
    inf_aver = adata_ref.varm['means_per_cluster_mu_fg'][[f'means_per_cluster_mu_fg_{i}'
                                    for i in adata_ref.uns['mod']['factor_names']]].copy()
else:
    inf_aver = adata_ref.var[[f'means_per_cluster_mu_fg_{i}'
                                    for i in adata_ref.uns['mod']['factor_names']]].copy()
inf_aver.columns = adata_ref.uns['mod']['factor_names']
inf_aver.iloc[0:5, 0:15]

In [ ]:
adata_file = f"{MinorInAmpullate_All_slide_run_name}/sp.h5ad"
adata_minor_major = sc.read_h5ad(adata_file)
mod = cell2location.models.Cell2location.load(f"{MinorInAmpullate_All_slide_run_name}", adata_minor_major)

In [ ]:
# add 5% quantile, representing confident cell abundance, 'at least this amount is present',
# to adata.obs with nice names for plotting
adata_minor_major.obs[adata_minor_major.uns['mod']['factor_names']] = adata_minor_major.obsm['q05_cell_abundance_w_sf']

Slide1

In [ ]:
slide_1 = select_slide(adata_minor_major, 'P18163_1001')

In [ ]:
# plot in spatial coordinates
with mpl.rc_context({'axes.facecolor':  'black',
                     'figure.figsize': [4.5, 5]}):
    sc.pl.spatial(slide_1, cmap='viridis',
                  color=['Minor_ZoneA_MiSpabd',
                         'Minor_ZoneB_MiSpc', 'Minor_ZoneC'],
                  ncols=4, size=1,
                  img_key='hires',
                  vmin=0, vmax=8,
                  save = 'MinorInAmpullate_Slide1_celltypes.pdf'
    )

Slide2

In [ ]:
slide_2 = select_slide(adata_minor_major, 'P18163_1002')
# plot in spatial coordinates
with mpl.rc_context({'axes.facecolor':  'black',
                     'figure.figsize': [4.5, 5]}):
    sc.pl.spatial(slide_2, cmap='viridis',
                  color=['Minor_ZoneA_MiSpabd',
                         'Minor_ZoneB_MiSpc', 'Minor_ZoneC'],
                  ncols=4, size=1,
                  img_key='hires',
                  vmin=0, vmax=6,
                  save = 'MinorInAmpullate_Slide2_celltypes.pdf'
    )

Slide3

In [ ]:
slide_3 = select_slide(adata_minor_major, 'P18163_1003')
# plot in spatial coordinates
with mpl.rc_context({'axes.facecolor':  'black',
                     'figure.figsize': [4.5, 5]}):
    sc.pl.spatial(slide_3, cmap='viridis',
                  color=['Minor_ZoneA_MiSpabd',
                         'Minor_ZoneB_MiSpc', 'Minor_ZoneC'],
                  ncols=4, size=1,
                  img_key='hires',
                  vmin=0, vmax=15,
                  save = 'MinorInAmpullate_Slide3_celltypes.pdf'
    )

Slide4

In [ ]:
slide_4 = select_slide(adata_minor_major, 'P18163_1004')
# plot in spatial coordinates
with mpl.rc_context({'axes.facecolor':  'black',
                     'figure.figsize': [4.5, 5]}):
    sc.pl.spatial(slide_4, cmap='viridis',
                  color=['Minor_ZoneA_MiSpabd',
                         'Minor_ZoneB_MiSpc', 'Minor_ZoneC'],
                  ncols=4, size=1,
                  img_key='hires',
                  vmin=0, vmax=3,
                  save = 'MinorInAmpullate_Slide4_celltypes.pdf'
    )

Slide5

In [ ]:
slide_5 = select_slide(adata_minor_major, 'P18163_1005')
# plot in spatial coordinates
with mpl.rc_context({'axes.facecolor':  'black',
                     'figure.figsize': [4.5, 5]}):
    sc.pl.spatial(slide_5, cmap='viridis',
                  color=['Minor_ZoneA_MiSpabd',
                         'Minor_ZoneB_MiSpc', 'Minor_ZoneC'],
                  ncols=4, size=1,
                  img_key='hires',
                  vmin=0, vmax=14,
                  save = 'MinorInAmpullate_Slide5_celltypes.pdf'
    )

Slide6

In [ ]:
slide_6 = select_slide(adata_minor_major, 'P22555_1001')
# plot in spatial coordinates
with mpl.rc_context({'axes.facecolor':  'black',
                     'figure.figsize': [4.5, 5]}):
    sc.pl.spatial(slide_6, cmap='viridis',
                  color=['Minor_ZoneA_MiSpabd',
                         'Minor_ZoneB_MiSpc', 'Minor_ZoneC'],
                  ncols=4, size=1,
                  img_key='hires',
                  vmin=0, vmax=10,
                  save = 'MinorInAmpullate_Slide6_celltypes.pdf'
    )

Slide7

In [ ]:
slide_7 = select_slide(adata_minor_major, 'P22555_1002')
# plot in spatial coordinates
with mpl.rc_context({'axes.facecolor':  'black',
                     'figure.figsize': [4.5, 5]}):
    sc.pl.spatial(slide_7, cmap='viridis',
                  color=['Minor_ZoneA_MiSpabd',
                         'Minor_ZoneB_MiSpc', 'Minor_ZoneC'],
                  ncols=4, size=1,
                  img_key='hires',
                  vmin=0, vmax=8.5,
                  save = 'MinorInAmpullate_Slide7_celltypes.pdf'
    )

Slide8

In [ ]:
slide_8 = select_slide(adata_minor_major, 'P22555_1003')
# plot in spatial coordinates
with mpl.rc_context({'axes.facecolor':  'black',
                     'figure.figsize': [4.5, 5]}):
    sc.pl.spatial(slide_8, cmap='viridis',
                  color=['Minor_ZoneA_MiSpabd',
                         'Minor_ZoneB_MiSpc', 'Minor_ZoneC'],
                  ncols=4, size=1,
                  img_key='hires',
                  vmin=0, vmax=11,
                  save = 'MinorInAmpullate_Slide8_celltypes.pdf'
    )

## Cell2location: spatial mapping -- 8slides -- MinorInMajor <a class="anchor" id="Cell2location:-spatial-mapping"></a>

<div class="alert alert-block alert-message">
<b>Find shared genes and prepare anndata.</b>
Subset both anndata and reference signatures:
</div>

In [ ]:
from os import makedirs
from os.path import exists

results_folder = str(RESULTS_DIR / 'Sub_Focus')

# create paths and names to results folders for reference regression and cell2location models
slide4_ref_run_name = f'{results_folder}/all_reference_signatures'
MinorInMajor_All_slide_run_name = f'{results_folder}/MinorInMajor_AllSlide_cell2location_map'

if not exists(slide4_ref_run_name):
  makedirs(slide4_ref_run_name)
if not exists(MinorInMajor_All_slide_run_name):
  makedirs(MinorInMajor_All_slide_run_name)

In [ ]:
# load reference single cell signature
adata_file = f"{slide4_ref_run_name}/sc_Minor.h5ad"
adata_ref = sc.read_h5ad(adata_file)
mod = cell2location.models.RegressionModel.load(f"{slide4_ref_run_name}", adata_ref)


In [ ]:
# export estimated expression in each cluster
if 'means_per_cluster_mu_fg' in adata_ref.varm.keys():
    inf_aver = adata_ref.varm['means_per_cluster_mu_fg'][[f'means_per_cluster_mu_fg_{i}'
                                    for i in adata_ref.uns['mod']['factor_names']]].copy()
else:
    inf_aver = adata_ref.var[[f'means_per_cluster_mu_fg_{i}'
                                    for i in adata_ref.uns['mod']['factor_names']]].copy()
inf_aver.columns = adata_ref.uns['mod']['factor_names']
inf_aver.iloc[0:5, 0:15]

In [ ]:
Marker_Genes = pd.read_csv(f'{DATA_DIR}/minor/Sub_Recluster_allmarkers_join_geneinfo_0.15.tsv', sep = '\t')
Marker_Genes['gene']

In [ ]:
# find shared genes and subset both anndata and reference signatures
intersect = np.intersect1d(adata_major.var.index, inf_aver.index)
#intersect = np.intersect1d(intersect, silks['geneName'])
intersect = np.intersect1d(intersect, Marker_Genes['gene'])
adata_major = adata_major[:, intersect].copy()
inf_aver = inf_aver.loc[intersect, :].copy()

# prepare anndata for cell2location model
cell2location.models.Cell2location.setup_anndata(adata=adata_major, batch_key="sample")

In [ ]:
# create and train the model
mod = cell2location.models.Cell2location(
    adata_major, cell_state_df=inf_aver,
    # the expected average cell abundance: tissue-dependent hyper-prior which can be estimated from paired histology:
    N_cells_per_location=5,
    # hyperparameter controlling normalisation of within-experiment variation in RNA detection: high technical variability: 20; low technical variability: 200
    detection_alpha=20,
    detection_mean_per_sample = False
)
mod.view_anndata_setup()

<div class="alert alert-block alert-message">
<b>Training cell2location:</b>
</div>

In [ ]:
# Reuse the saved cell2location model if it already exists (delete the folder to retrain)
if (MinorInMajor_All_slide_run_name and (Path(MinorInMajor_All_slide_run_name) / "model.pt").exists()):
    mod = cell2location.models.Cell2location.load(MinorInMajor_All_slide_run_name, adata_major)
    print("Loaded cached model from", MinorInMajor_All_slide_run_name)
else:
    mod.train(max_epochs=5500,
              # train using full data (batch_size=None)
              batch_size=None,
              # use all data points in training because we need to estimate cell abundance at all locations
              train_size=1)

    # plot ELBO loss history during training, removing first 100 epochs from the plot
    mod.plot_history(1000)
    plt.legend(labels=['full data training']);


<div class="alert alert-block alert-message">
<b>Exporting estimated posterior distributions of cell abundance and saving results:</b>
</div>

In [ ]:
# In this section, we export the estimated cell abundance (summary of the posterior distribution).
adata_spatial = mod.export_posterior(
    adata_major, sample_kwargs={'num_samples': 1000, 'batch_size': mod.adata.n_obs}
)

# Save model
mod.save(f"{MinorInMajor_All_slide_run_name}", overwrite=True)

# mod = cell2location.models.Cell2location.load(f"{run_name}", adata_major)

# Save anndata object with results
adata_file = f"{MinorInMajor_All_slide_run_name}/sp.h5ad"
adata_major.write(adata_file)
adata_file

<div class="alert alert-block alert-message">
<b>Assessing mapping quality.</b>
Examine reconstruction accuracy to assess if there are any issues with mapping.
The plot should be roughly diagonal, strong deviations will signal problems that need to be investigated.
</div>

In [ ]:
mod.plot_QC()

In [ ]:
fig = mod.plot_spatial_QC_across_batches()
fig.savefig("spatial_QC_across_batches.pdf")

#### Visualising cell abundance in spatial coordinates <a class="anchor" id="Visualising-cell-abundance-in-spatial-coordinates"></a>

<div class="alert alert-info">
Note
    
We use 5% quantile of the posterior distribution, representing the value of cell abundance that the model has high confidence in (aka 'at least this amount is present').
</div>

In [ ]:
# load reference single cell signature
adata_file = f"{slide4_ref_run_name}/sc_Minor.h5ad"
adata_ref = sc.read_h5ad(adata_file)
mod = cell2location.models.RegressionModel.load(f"{slide4_ref_run_name}", adata_ref)


In [ ]:
# export estimated expression in each cluster
if 'means_per_cluster_mu_fg' in adata_ref.varm.keys():
    inf_aver = adata_ref.varm['means_per_cluster_mu_fg'][[f'means_per_cluster_mu_fg_{i}'
                                    for i in adata_ref.uns['mod']['factor_names']]].copy()
else:
    inf_aver = adata_ref.var[[f'means_per_cluster_mu_fg_{i}'
                                    for i in adata_ref.uns['mod']['factor_names']]].copy()
inf_aver.columns = adata_ref.uns['mod']['factor_names']
inf_aver.iloc[0:5, 0:15]

In [ ]:
adata_file = f"{MinorInMajor_All_slide_run_name}/sp.h5ad"
adata_major = sc.read_h5ad(adata_file)
mod = cell2location.models.Cell2location.load(f"{MinorInMajor_All_slide_run_name}", adata_major)

In [ ]:
# add 5% quantile, representing confident cell abundance, 'at least this amount is present',
# to adata.obs with nice names for plotting
adata_major.obs[adata_major.uns['mod']['factor_names']] = adata_major.obsm['q05_cell_abundance_w_sf']

In [ ]:
sample_data

Slide1

In [ ]:
slide_1 = select_slide(adata_major, 'P18163_1001')

In [ ]:
# plot in spatial coordinates
with mpl.rc_context({'axes.facecolor':  'black',
                     'figure.figsize': [4.5, 5]}):
    sc.pl.spatial(slide_1, cmap='viridis',
                  color=['Minor_ZoneA_MiSpabd',
                         'Minor_ZoneB_MiSpc', 'Minor_ZoneC'],
                  ncols=4, size=1,
                  img_key='hires',
                  vmin=0, vmax=9,
                  save = 'MinorInMajor_Slide1_celltypes.pdf'
    )

Slide2

In [ ]:
slide_2 = select_slide(adata_major, 'P18163_1002')
# plot in spatial coordinates
with mpl.rc_context({'axes.facecolor':  'black',
                     'figure.figsize': [4.5, 5]}):
    sc.pl.spatial(slide_2, cmap='viridis',
                  color=['Minor_ZoneA_MiSpabd',
                         'Minor_ZoneB_MiSpc', 'Minor_ZoneC'],
                  ncols=4, size=1,
                  img_key='hires',
                  vmin=0, vmax=8,
                  save = 'MinorInMajor_Slide2_celltypes.pdf'
    )

Slide3

In [ ]:
slide_3 = select_slide(adata_major, 'P18163_1003')
# plot in spatial coordinates
with mpl.rc_context({'axes.facecolor':  'black',
                     'figure.figsize': [4.5, 5]}):
    sc.pl.spatial(slide_3, cmap='viridis',
                  color=['Minor_ZoneA_MiSpabd',
                         'Minor_ZoneB_MiSpc', 'Minor_ZoneC'],
                  ncols=4, size=1,
                  img_key='hires',
                  vmin=0, vmax=20,
                  save = 'MinorInMajor_Slide3_celltypes.pdf'
    )

Slide4

In [ ]:
slide_4 = select_slide(adata_major, 'P18163_1004')
# plot in spatial coordinates
with mpl.rc_context({'axes.facecolor':  'black',
                     'figure.figsize': [4.5, 5]}):
    sc.pl.spatial(slide_4, cmap='viridis',
                  color=['Minor_ZoneA_MiSpabd',
                         'Minor_ZoneB_MiSpc', 'Minor_ZoneC'],
                  ncols=4, size=1,
                  img_key='hires',
                  vmin=0, vmax=8,
                  save = 'MinorInMajor_Slide4_celltypes.pdf'
    )

Slide5

In [ ]:
slide_5 = select_slide(adata_major, 'P18163_1005')
# plot in spatial coordinates
with mpl.rc_context({'axes.facecolor':  'black',
                     'figure.figsize': [4.5, 5]}):
    sc.pl.spatial(slide_5, cmap='viridis',
                  color=['Minor_ZoneA_MiSpabd',
                         'Minor_ZoneB_MiSpc', 'Minor_ZoneC'],
                  ncols=4, size=1,
                  img_key='hires',
                  vmin=0, vmax=16,
                  save = 'MinorInMajor_Slide5_celltypes.pdf'
    )

Slide6

In [ ]:
slide_6 = select_slide(adata_major, 'P22555_1001')
# plot in spatial coordinates
with mpl.rc_context({'axes.facecolor':  'black',
                     'figure.figsize': [4.5, 5]}):
    sc.pl.spatial(slide_6, cmap='viridis',
                  color=['Minor_ZoneA_MiSpabd',
                         'Minor_ZoneB_MiSpc', 'Minor_ZoneC'],
                  ncols=4, size=1,
                  img_key='hires',
                  vmin=0, vmax=10,
                  save = 'MinorInMajor_Slide6_celltypes.pdf'
    )

Slide7

In [ ]:
slide_7 = select_slide(adata_major, 'P22555_1002')
# plot in spatial coordinates
with mpl.rc_context({'axes.facecolor':  'black',
                     'figure.figsize': [4.5, 5]}):
    sc.pl.spatial(slide_7, cmap='viridis',
                  color=['Minor_ZoneA_MiSpabd',
                         'Minor_ZoneB_MiSpc', 'Minor_ZoneC'],
                  ncols=4, size=1,
                  img_key='hires',
                  vmin=0, vmax=10,
                  save = 'MinorInMajor_Slide7_celltypes.pdf'
    )

Slide8

In [ ]:
slide_8 = select_slide(adata_major, 'P22555_1003')
# plot in spatial coordinates
with mpl.rc_context({'axes.facecolor':  'black',
                     'figure.figsize': [4.5, 5]}):
    sc.pl.spatial(slide_8, cmap='viridis',
                  color=['Minor_ZoneA_MiSpabd',
                         'Minor_ZoneB_MiSpc', 'Minor_ZoneC'],
                  ncols=4, size=1,
                  img_key='hires',
                  vmin=0, vmax=10,
                  save = 'MinorInMajor_Slide8_celltypes.pdf'
    )

## Cell2location: spatial mapping -- 8slides -- MinorInMinor <a class="anchor" id="Visualising-cell-abundance-in-spatial-coordinates"></a>


In [ ]:
from os import makedirs
from os.path import exists

results_folder = str(RESULTS_DIR / 'Sub_Focus')

# create paths and names to results folders for reference regression and cell2location models
slide4_ref_run_name = f'{results_folder}/all_reference_signatures'
MinorInMinor_All_slide_run_name = f'{results_folder}/MinorInMinor_AllSlide_cell2location_map'

if not exists(slide4_ref_run_name):
  makedirs(slide4_ref_run_name)
if not exists(MinorInMinor_All_slide_run_name):
  makedirs(MinorInMinor_All_slide_run_name)

In [ ]:
# load reference single cell signature
adata_file = f"{slide4_ref_run_name}/sc_Minor.h5ad"
adata_ref = sc.read_h5ad(adata_file)
mod = cell2location.models.RegressionModel.load(f"{slide4_ref_run_name}", adata_ref)


In [ ]:
# export estimated expression in each cluster
if 'means_per_cluster_mu_fg' in adata_ref.varm.keys():
    inf_aver = adata_ref.varm['means_per_cluster_mu_fg'][[f'means_per_cluster_mu_fg_{i}'
                                    for i in adata_ref.uns['mod']['factor_names']]].copy()
else:
    inf_aver = adata_ref.var[[f'means_per_cluster_mu_fg_{i}'
                                    for i in adata_ref.uns['mod']['factor_names']]].copy()
inf_aver.columns = adata_ref.uns['mod']['factor_names']
inf_aver.iloc[0:5, 0:15]

In [ ]:
adata_minor = sc.read_h5ad(f"{DATA_DIR}/spatial/all_8_slides_minor_data.h5ad")

In [ ]:
Marker_Genes = pd.read_csv(f'{DATA_DIR}/minor/Sub_Recluster_allmarkers_join_geneinfo_0.15.tsv', sep = '\t')
Marker_Genes['gene']

In [ ]:
# find shared genes and subset both anndata and reference signatures
intersect = np.intersect1d(adata_minor.var.index, inf_aver.index)
#intersect = np.intersect1d(intersect, silks['geneName'])
intersect = np.intersect1d(intersect, Marker_Genes['gene'])
adata_minor = adata_minor[:, intersect].copy()
inf_aver = inf_aver.loc[intersect, :].copy()

# prepare anndata for cell2location model
cell2location.models.Cell2location.setup_anndata(adata=adata_minor, batch_key="sample")

In [ ]:
# create and train the model
mod = cell2location.models.Cell2location(
    adata_minor, cell_state_df=inf_aver,
    # the expected average cell abundance: tissue-dependent hyper-prior which can be estimated from paired histology:
    N_cells_per_location=5,
    # hyperparameter controlling normalisation of within-experiment variation in RNA detection: high technical variability: 20; low technical variability: 200
    detection_alpha=20,
    detection_mean_per_sample = False
)
mod.view_anndata_setup()

<div class="alert alert-block alert-message">
<b>Training cell2location:</b>
</div>

In [ ]:
# Reuse the saved cell2location model if it already exists (delete the folder to retrain)
if (MinorInMinor_All_slide_run_name and (Path(MinorInMinor_All_slide_run_name) / "model.pt").exists()):
    mod = cell2location.models.Cell2location.load(MinorInMinor_All_slide_run_name, adata_minor)
    print("Loaded cached model from", MinorInMinor_All_slide_run_name)
else:
    mod.train(max_epochs=5500,
              # train using full data (batch_size=None)
              batch_size=None,
              # use all data points in training because we need to estimate cell abundance at all locations
              train_size=1)

    # plot ELBO loss history during training, removing first 100 epochs from the plot
    mod.plot_history(1000)
    plt.legend(labels=['full data training']);


<div class="alert alert-block alert-message">
<b>Exporting estimated posterior distributions of cell abundance and saving results:</b>
</div>

In [ ]:
# In this section, we export the estimated cell abundance (summary of the posterior distribution).
adata_spatial = mod.export_posterior(
    adata_minor, sample_kwargs={'num_samples': 1000, 'batch_size': mod.adata.n_obs}
)

# Save model
mod.save(f"{MinorInMinor_All_slide_run_name}", overwrite=True)

# mod = cell2location.models.Cell2location.load(f"{run_name}", adata_minor)

# Save anndata object with results
adata_file = f"{MinorInMinor_All_slide_run_name}/sp.h5ad"
adata_minor.write(adata_file)
adata_file

<div class="alert alert-block alert-message">
<b>Assessing mapping quality.</b>
Examine reconstruction accuracy to assess if there are any issues with mapping.
The plot should be roughly diagonal, strong deviations will signal problems that need to be investigated.
</div>

In [ ]:
mod.plot_QC()

In [ ]:
fig = mod.plot_spatial_QC_across_batches()
fig.savefig("MinorInMinor_spatial_QC_across_batches.pdf")

#### Visualising cell abundance in spatial coordinates <a class="anchor" id="Visualising-cell-abundance-in-spatial-coordinates"></a>

<div class="alert alert-info">
Note
    
We use 5% quantile of the posterior distribution, representing the value of cell abundance that the model has high confidence in (aka 'at least this amount is present').
</div>

In [ ]:
# load reference single cell signature
adata_file = f"{slide4_ref_run_name}/sc_Minor.h5ad"
adata_ref = sc.read_h5ad(adata_file)
mod = cell2location.models.RegressionModel.load(f"{slide4_ref_run_name}", adata_ref)


In [ ]:
# export estimated expression in each cluster
if 'means_per_cluster_mu_fg' in adata_ref.varm.keys():
    inf_aver = adata_ref.varm['means_per_cluster_mu_fg'][[f'means_per_cluster_mu_fg_{i}'
                                    for i in adata_ref.uns['mod']['factor_names']]].copy()
else:
    inf_aver = adata_ref.var[[f'means_per_cluster_mu_fg_{i}'
                                    for i in adata_ref.uns['mod']['factor_names']]].copy()
inf_aver.columns = adata_ref.uns['mod']['factor_names']
inf_aver.iloc[0:5, 0:15]

In [ ]:
adata_file = f"{MinorInMinor_All_slide_run_name}/sp.h5ad"
adata_minor = sc.read_h5ad(adata_file)
mod = cell2location.models.Cell2location.load(f"{MinorInMinor_All_slide_run_name}", adata_minor)

In [ ]:
# add 5% quantile, representing confident cell abundance, 'at least this amount is present',
# to adata.obs with nice names for plotting
adata_minor.obs[adata_minor.uns['mod']['factor_names']] = adata_minor.obsm['q05_cell_abundance_w_sf']

In [ ]:
sample_data

Slide1

In [ ]:
slide_1 = select_slide(adata_minor, 'P18163_1001')

In [ ]:
# plot in spatial coordinates
with mpl.rc_context({'axes.facecolor':  'black',
                     'figure.figsize': [4.5, 5]}):
    sc.pl.spatial(slide_1, cmap='viridis',
                  color=['Minor_ZoneA_MiSpabd',
                         'Minor_ZoneB_MiSpc', 'Minor_ZoneC'],
                  ncols=4, size=1,
                  img_key='hires',
                  vmin=0, vmax=6,
                  save = 'MinorInMinor_Slide1_celltypes.pdf'
    )

Slide2

In [ ]:
slide_2 = select_slide(adata_minor, 'P18163_1002')
# plot in spatial coordinates
with mpl.rc_context({'axes.facecolor':  'black',
                     'figure.figsize': [4.5, 5]}):
    sc.pl.spatial(slide_2, cmap='viridis',
                  color=['Minor_ZoneA_MiSpabd',
                         'Minor_ZoneB_MiSpc', 'Minor_ZoneC'],
                  ncols=4, size=1,
                  img_key='hires',
                  vmin=0, vmax=6,
                  save = 'MinorInMinor_Slide2_celltypes.pdf'
    )

Slide3

In [ ]:
slide_3 = select_slide(adata_minor, 'P18163_1003')
# plot in spatial coordinates
with mpl.rc_context({'axes.facecolor':  'black',
                     'figure.figsize': [4.5, 5]}):
    sc.pl.spatial(slide_3, cmap='viridis',
                  color=['Minor_ZoneA_MiSpabd',
                         'Minor_ZoneB_MiSpc', 'Minor_ZoneC'],
                  ncols=4, size=1,
                  img_key='hires',
                  #vmin=0, vmax=20,
                  save = 'MinorInMinor_Slide3_celltypes.pdf'
    )

Slide4

In [ ]:
slide_4 = select_slide(adata_minor, 'P18163_1004')
# plot in spatial coordinates
with mpl.rc_context({'axes.facecolor':  'black',
                     'figure.figsize': [4.5, 5]}):
    sc.pl.spatial(slide_4, cmap='viridis',
                  color=['Minor_ZoneA_MiSpabd',
                         'Minor_ZoneB_MiSpc', 'Minor_ZoneC'],
                  ncols=4, size=1,
                  img_key='hires',
                  vmin=0, vmax=5,
                  save = 'MinorInMinor_Slide4_celltypes.pdf'
    )

Slide5

In [ ]:
slide_5 = select_slide(adata_minor, 'P18163_1005')
# plot in spatial coordinates
with mpl.rc_context({'axes.facecolor':  'black',
                     'figure.figsize': [4.5, 5]}):
    sc.pl.spatial(slide_5, cmap='viridis',
                  color=['Minor_ZoneA_MiSpabd',
                         'Minor_ZoneB_MiSpc', 'Minor_ZoneC'],
                  ncols=4, size=1,
                  img_key='hires',
                  vmin=0, vmax=7,
                  save = 'MinorInMinor_Slide5_celltypes.pdf'
    )

Slide6

In [ ]:
slide_6 = select_slide(adata_minor, 'P22555_1001')
# plot in spatial coordinates
with mpl.rc_context({'axes.facecolor':  'black',
                     'figure.figsize': [4.5, 5]}):
    sc.pl.spatial(slide_6, cmap='viridis',
                  color=['Minor_ZoneA_MiSpabd',
                         'Minor_ZoneB_MiSpc', 'Minor_ZoneC'],
                  ncols=4, size=1,
                  img_key='hires',
                  vmin=0, vmax=9,
                  save = 'MinorInMinor_Slide6_celltypes.pdf'
    )

Slide7

In [ ]:
slide_7 = select_slide(adata_minor, 'P22555_1002')
# plot in spatial coordinates
with mpl.rc_context({'axes.facecolor':  'black',
                     'figure.figsize': [4.5, 5]}):
    sc.pl.spatial(slide_7, cmap='viridis',
                  color=['Minor_ZoneA_MiSpabd',
                         'Minor_ZoneB_MiSpc', 'Minor_ZoneC'],
                  ncols=4, size=1,
                  img_key='hires',
                  vmin=0, vmax=6,
                  save = 'MinorInMinor_Slide7_celltypes.pdf'
    )

Slide8

In [ ]:
slide_8 = select_slide(adata_minor, 'P22555_1003')
# plot in spatial coordinates
with mpl.rc_context({'axes.facecolor':  'black',
                     'figure.figsize': [4.5, 5]}):
    sc.pl.spatial(slide_8, cmap='viridis',
                  color=['Minor_ZoneA_MiSpabd',
                         'Minor_ZoneB_MiSpc', 'Minor_ZoneC'],
                  ncols=4, size=1,
                  img_key='hires',
                  vmin=0, vmax=4.5,
                  save = 'MinorInMinor_Slide8_celltypes.pdf'
    )